##  1 - Đường có lề rộng nhất và vectơ hỗ trợ

In [ ]:
# -*- coding: utf-8 -*-


import numpy as np
from sklearn.datasets import make_blobs
from sklearn.svm import SVC

# 24 điểm, hai cụm tách hẳn nhau
X, y = make_blobs(n_samples=24, centers=[[2, 2], [6, 6]],
                  cluster_std=0.9, random_state=0)
y = 1 - y  # lớp 1 là cụm dưới bên trái, giống hình trên slide

# C rất lớn nghĩa là không cho điểm nào nằm trong lề
svm = SVC(kernel="linear", C=1000)
svm.fit(X, y)

w = svm.coef_[0]
b = svm.intercept_[0]
print(f"w1 = {w[0]:.3f}, w2 = {w[1]:.3f}, b = {b:.3f}")
do_dai_w = np.sqrt(w[0] ** 2 + w[1] ** 2)
print(f"Do rong cua le = 2 / {do_dai_w:.4f} = {2 / do_dai_w:.4f}")
print()

# Các vectơ hỗ trợnằm đúng trên hai mép lề, nơi f(x) bằng 1 hoặc -1
print("So vecto ho tro:", len(svm.support_))
for i in svm.support_:
    f = svm.decision_function(X[i:i + 1])[0]
    print(f"diem {i:2d}: x1 = {X[i, 0]:.2f}, x2 = {X[i, 1]:.2f},"
          f" lop {y[i]}, f(x) = {f:+.3f}")
print()

# Những điểm khác nằm ngoài lề, nên |f(x)| lớn hơn 1
f_tat_ca = svm.decision_function(X)
ngoai = np.setdiff1d(np.arange(len(X)), svm.support_)
nho_nhat = np.abs(f_tat_ca[ngoai]).min()
print(f"|f(x)| nho nhat cua cac diem khac: {nho_nhat:.3f}")
print()

# Học lại chỉ với các vectơ hỗ trợ
chi_sv = SVC(kernel="linear", C=1000)
chi_sv.fit(X[svm.support_], y[svm.support_])
w2 = chi_sv.coef_[0]
print("Hoc lai chi voi vecto ho tro:")
b2 = chi_sv.intercept_[0]
print(f"w1 = {w2[0]:.3f}, w2 = {w2[1]:.3f}, b = {b2:.3f}")
print()

# Dự đoán cho hai điểm mới
moi = np.array([[3.0, 3.0], [5.0, 4.5]])
for diem, f, lop in zip(moi, svm.decision_function(moi),
                        svm.predict(moi)):
    print(f"Diem ({diem[0]}, {diem[1]}): f(x) = {f:+.3f}, doan lop {lop}")


w1 = -0.831, w2 = -0.355, b = 4.820
Do rong cua le = 2 / 0.9039 = 2.2126

So vecto ho tro: 3
diem 18: x1 = 5.20, x2 = 4.22, lop 0, f(x) = -1.000
diem 10: x1 = 2.88, x2 = 4.02, lop 1, f(x) = +1.000
diem 13: x1 = 3.59, x2 = 2.36, lop 1, f(x) = +1.000

|f(x)| nho nhat cua cac diem khac: 1.059

Hoc lai chi voi vecto ho tro:
w1 = -0.831, w2 = -0.355, b = 4.821

Diem (3.0, 3.0): f(x) = +1.262, doan lop 1
Diem (5.0, 4.5): f(x) = -0.933, doan lop 0


##  2 - Lề mềm và tham số C

In [ ]:
# -*- coding: utf-8 -*-

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

df = pd.read_csv("data/sinh_vien.csv")
X = df[["gio_on", "diem_giua_ky"]]
y = df["qua_mon"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

print("Tap hoc co", len(X_tr), "ban")
print("C        so vecto HT   do rong le   tap hoc   tap kiem tra")
for C in [0.01, 0.1, 1, 10, 100]:
    mh = make_pipeline(StandardScaler(),
                       SVC(kernel="linear", C=C))
    mh.fit(X_tr, y_tr)
    svm = mh[-1]  # bước cuối của pipeline là SVM
    le = 2 / np.linalg.norm(svm.coef_[0])
    a_hoc = accuracy_score(y_tr, mh.predict(X_tr))
    a_kt = accuracy_score(y_te, mh.predict(X_te))
    print(f"{C:6}     {len(svm.support_):6d}        {le:6.2f}       "
          f"{a_hoc:.4f}       {a_kt:.4f}")
print()

# Bộ sinh viên mở rộng có thêm cột chi tiêu, tính bằng nghìn đồng
df3 = pd.read_csv("data/sinh_vien_mo_rong.csv")
X3 = df3[["gio_on", "diem_giua_ky", "chi_tieu"]]
y3 = df3["qua_mon"]
X3_tr, X3_te, y3_tr, y3_te = train_test_split(
    X3, y3, test_size=0.25, random_state=17, stratify=y3
)
tho = SVC().fit(X3_tr, y3_tr)
chuan = make_pipeline(StandardScaler(), SVC()).fit(X3_tr, y3_tr)
print("Ba cot, kernel RBF mac dinh")
print(f"Doan bua ca lop qua : {y3_te.mean():.4f}")
print(f"Khong chuan hoa: "
      f"{accuracy_score(y3_te, tho.predict(X3_te)):.4f}")
print(f"Co chuan hoa: "
      f"{accuracy_score(y3_te, chuan.predict(X3_te)):.4f}")


Tap hoc co 90 ban
C        so vecto HT   do rong le   tap hoc   tap kiem tra
  0.01         74          3.84       0.8556       0.8667
   0.1         44          1.70       0.8889       0.9000
     1         31          1.29       0.8778       0.9000
    10         28          1.06       0.9000       0.9000
   100         27          1.05       0.9000       0.9000

Ba cot, kernel RBF mac dinh
Doan bua ca lop qua : 0.6000
Khong chuan hoa: 0.6000
Co chuan hoa: 0.9000


##  3 - Kernel và biên cong

In [ ]:
# -*- coding: utf-8 -*-

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

# Phần 1: dữ liệu một chiều, lớp 1 nằm giữa, lớp 0 ở hai đầu
x = np.array([-4.0, -3.4, -2.8, -2.2, -1.2, -0.7, -0.2,
              0.3, 0.8, 1.3, 2.3, 2.9, 3.5, 4.0])
y = (np.abs(x) < 1.8).astype(int)

# Thử mọi ngưỡng: đoán lớp 1 khi x nhỏ hơn hoặc bằng ngưỡng
tot = max(((x <= t).astype(int) == y).mean() for t in x)
print(f"Mot nguong tren truc x, tot nhat dung {tot:.4f}")

# Thêm cột x binh phuong rồi dùng SVM tuyến tính
X_moi = np.c_[x, x ** 2]
svm = SVC(kernel="linear", C=1000).fit(X_moi, y)
print(f"Them cot x^2, SVM tuyen tinh dung {svm.score(X_moi, y):.4f}")
print()

# Phần 2: hai vầng trăng, để kernel tự nâng chiều
df = pd.read_csv("data/hai_vang_trang.csv")
X = df[["x1", "x2"]]
y = df["nhan"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

mo_hinh = {
    "SVM kernel tuyen tinh": SVC(kernel="linear"),
    "SVM kernel da thuc bac 3": SVC(kernel="poly", degree=3),
    "SVM kernel RBF": SVC(kernel="rbf"),
    "Hoi quy logistic": LogisticRegression(),
    "KNN, k = 13": KNeighborsClassifier(n_neighbors=13),
}
print("Hai vang trang, accuracy tren tap kiem tra")
for ten, mh in mo_hinh.items():
    mh.fit(X_tr, y_tr)
    acc = accuracy_score(y_te, mh.predict(X_te))
    print(f"{ten:26s}  {acc:.4f}")
print()

rbf = mo_hinh["SVM kernel RBF"]
print("So vecto ho tro cua kernel RBF:", len(rbf.support_),
      "tren", len(X_tr), "diem hoc")


Mot nguong tren truc x, tot nhat dung 0.7143
Them cot x^2, SVM tuyen tinh dung 1.0000

Hai vang trang, accuracy tren tap kiem tra
SVM kernel tuyen tinh       0.8667
SVM kernel da thuc bac 3    0.8667
SVM kernel RBF              0.9200
Hoi quy logistic            0.8533
KNN, k = 13                 0.9200

So vecto ho tro cua kernel RBF: 89 tren 225 diem hoc


##  4 - Chọn C và Gamma

In [ ]:
# -*- coding: utf-8 -*-

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.svm import SVC

df = pd.read_csv("data/hai_vang_trang.csv")
X = df[["x1", "x2"]]
y = df["nhan"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

print("gamma     tap hoc    tap kiem tra    so vecto HT")
for g in [0.1, 1, 10, 100]:
    mh = SVC(kernel="rbf", gamma=g).fit(X_tr, y_tr)
    a_hoc = accuracy_score(y_tr, mh.predict(X_tr))
    a_kt = accuracy_score(y_te, mh.predict(X_te))
    print(f"{g:5}       {a_hoc:.4f}      {a_kt:.4f}         "
          f"{len(mh.support_)}")
print()

# Dò lưới: thửmọi cặp bằng kiểm định chéo 5 phần, chỉ trên tập học
luoi = {"C": [0.1, 1, 10, 100], "gamma": [0.01, 0.1, 1, 10]}
tim = GridSearchCV(SVC(kernel="rbf"), luoi, cv=5)
tim.fit(X_tr, y_tr)

bang = pd.DataFrame(tim.cv_results_["params"])
bang["diem"] = tim.cv_results_["mean_test_score"]
print("Diem kiem dinh cheo, hang la C, cot la gamma")
print(bang.pivot(index="C", columns="gamma",
                 values="diem").round(4).to_string())
print()

print("Cap tot nhat:", tim.best_params_)
print(f"Diem kiem dinh cheo: {tim.best_score_:.4f}")
# GridSearchCV đã tự học lại với cặp tốt nhất trên toàn tập học
print(f"Accuracy tren tap kiem tra: {tim.score(X_te, y_te):.4f}")


gamma     tap hoc    tap kiem tra    so vecto HT
  0.1       0.8356      0.8533         113
    1       0.8711      0.9200         88
   10       0.8978      0.9067         128
  100       0.9422      0.8133         217



Diem kiem dinh cheo, hang la C, cot la gamma
gamma   0.01    0.10    1.00    10.00
C                                    
0.1    0.6578  0.8044  0.8444  0.8578
1.0    0.8044  0.8178  0.8667  0.8667
10.0   0.8178  0.8400  0.8711  0.8311
100.0  0.8178  0.8533  0.8711  0.7467

Cap tot nhat: {'C': 10, 'gamma': 1}
Diem kiem dinh cheo: 0.8711
Accuracy tren tap kiem tra: 0.9200


##  5 - Nhận dạng chữ số viết tay

In [5]:
# -*- coding: utf-8 -*-
"""Bước 5: nhận dạng chữ sốviết tay bằng SVM."""

import numpy as np
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

X, y = load_digits(return_X_y=True)
print("Kich thuoc X:", X.shape)
print("Do sang nho nhat va lon nhat:", X.min(), X.max())
print("So anh moi chu so:", np.bincount(y))
print()

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

svm = SVC(kernel="rbf").fit(X_tr, y_tr)
du_doan = svm.predict(X_te)
print(f"SVM kernel RBF: accuracy {accuracy_score(y_te, du_doan):.4f}")
print("So vecto ho tro:", len(svm.support_), "tren", len(X_tr), "anh")
sai = np.flatnonzero(du_doan != y_te)
print("So anh doan sai:", len(sai), "tren", len(y_te))
for i in sai:
    print(f"anh {i:3d}: that la {y_te[i]}, doan la {du_doan[i]}")
print()

# So với các mô hình đã học
mo_hinh = {
    "SVM co chuan hoa": make_pipeline(StandardScaler(), SVC()),
    "KNN, k = 5": KNeighborsClassifier(),
    "Logistic co chuan hoa": make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=2000)),
}
print("Cac mo hinh khac, accuracy tren tap kiem tra")
for ten, mh in mo_hinh.items():
    mh.fit(X_tr, y_tr)
    acc = accuracy_score(y_te, mh.predict(X_te))
    print(f"{ten:22s} {acc:.4f}")


Kich thuoc X: (1797, 64)
Do sang nho nhat va lon nhat: 0.0 16.0
So anh moi chu so: [178 182 177 183 181 182 181 179 174 180]

SVM kernel RBF: accuracy 0.9911
So vecto ho tro: 626 tren 1347 anh
So anh doan sai: 4 tren 450
anh   6: that la 9, doan la 3
anh  18: that la 8, doan la 1
anh 161: that la 4, doan la 8
anh 428: that la 7, doan la 9

Cac mo hinh khac, accuracy tren tap kiem tra
SVM co chuan hoa       0.9778
KNN, k = 5             0.9867
Logistic co chuan hoa  0.9711


## Bài tập

### Bài tập 1 - Tính tay khoảng cách tới ranh giới

In [6]:
import numpy as np
from sklearn.datasets import make_blobs
from sklearn.svm import SVC

X, y = make_blobs(n_samples=24, centers=[[2, 2], [6, 6]],
                  cluster_std=0.9, random_state=0)
y = 1 - y
svm = SVC(kernel='linear', C=1000).fit(X, y)
w = svm.coef_[0]
b = svm.intercept_[0]
do_dai = np.sqrt(w[0] ** 2 + w[1] ** 2)
print(f'w1={w[0]:.4f}, w2={w[1]:.4f}, b={b:.4f}')
print(f'Do rong le = {2 / do_dai:.4f}, nua le = {1 / do_dai:.4f}')

# Tự tính bằng phép nhân và cộng, không dùng decision_function
cac_diem = [np.array([3.0, 3.0]), np.array([5.0, 4.5])]
for p in cac_diem:
    fx = w[0] * p[0] + w[1] * p[1] + b
    khoang_cach = abs(fx) / do_dai
    print(f'Diem ({p[0]}, {p[1]}): f(x)={fx:+.4f}, khoang cach={khoang_cach:.4f}')

print('Ba vecto ho tro:')
for idx in svm.support_:
    p = X[idx]
    fx = w[0] * p[0] + w[1] * p[1] + b
    print(f'  Diem {idx:2d}: f(x)={fx:+.4f}, khoang cach={abs(fx)/do_dai:.4f}')


w1=-0.8314, w2=-0.3549, b=4.8204
Do rong le = 2.2126, nua le = 1.1063
Diem (3.0, 3.0): f(x)=+1.2617, khoang cach=1.3958
Diem (5.0, 4.5): f(x)=-0.9333, khoang cach=1.0325
Ba vecto ho tro:
  Diem 18: f(x)=-1.0001, khoang cach=1.1064
  Diem 10: f(x)=+0.9999, khoang cach=1.1062
  Diem 13: f(x)=+1.0002, khoang cach=1.1065


**Nhận xét**

- Hai điểm mới cách đường ranh giới khoảng 1,3958 và 1,0325. Ba vectơ hỗ trợ cách ranh giới khoảng 1,1063, bằng một nửa lề rộng 2,2126.

### Bài tập 2 - Thêm một điểm vào SVM

In [7]:
import numpy as np
from sklearn.datasets import make_blobs
from sklearn.svm import SVC

X, y = make_blobs(n_samples=24, centers=[[2, 2], [6, 6]],
                  cluster_std=0.9, random_state=0)
y = 1 - y

def hoc_svm(X, y):
    mh = SVC(kernel='linear', C=1000)
    mh.fit(X, y)
    return mh

goc = hoc_svm(X, y)
print('Goc:', np.round(goc.coef_[0], 4), round(goc.intercept_[0], 4))
for p in [(-1.0, -1.0), (4.5, 4.0)]:
    X_them = np.vstack([X, np.array([p])])
    y_them = np.r_[y, 1]
    m = hoc_svm(X_them, y_them)
    doi_w = np.linalg.norm(m.coef_[0] - goc.coef_[0])
    doi_b = abs(m.intercept_[0] - goc.intercept_[0])
    print(f'Them {p}: w={np.round(m.coef_[0], 4)}, b={m.intercept_[0]:.4f}')
    print(f'  Chuyen dich: w={doi_w:.5f}, b={doi_b:.5f}, so vector ho tro={len(m.support_)}')
    print('  Ranh gioi thay doi ro:', doi_w > 0.01 or doi_b > 0.01)


Goc: [-0.8314 -0.3549] 4.8204
Them (-1.0, -1.0): w=[-0.8314 -0.3549], b=4.8204
  Chuyen dich: w=0.00000, b=0.00000, so vector ho tro=3
  Ranh gioi thay doi ro: False
Them (4.5, 4.0): w=[-2.603  -0.8068], b=15.9409
  Chuyen dich: w=1.82840, b=11.12058, so vector ho tro=2
  Ranh gioi thay doi ro: True


**Nhận xét**

- Thêm điểm (-1, -1) không làm thay đổi ranh giới. Thêm điểm (4.5, 4) làm thay đổi rõ vì điểm này nằm gần vùng cần phân chia và ảnh hưởng đến vectơ hỗ trợ.

### Bài tập 3 - Thử kernel đa thức

In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

df = pd.read_csv('data/hai_vang_trang.csv')
X = df[['x1', 'x2']]
y = df['nhan']
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

ket_qua = []
for bac in [2, 3, 4]:
    pipe = make_pipeline(StandardScaler(), SVC(kernel='poly', degree=bac))
    tim = GridSearchCV(pipe, {'svc__C': [0.1, 1, 10, 100]}, cv=5)
    tim.fit(X_tr, y_tr)
    ket_qua.append((bac, tim.best_params_['svc__C'], tim.best_score_, tim.score(X_te, y_te)))
    print(f'Poly bac {bac}: C={tim.best_params_["svc__C"]}, CV={tim.best_score_:.4f}, Test={tim.score(X_te, y_te):.4f}')

best_poly = max(ket_qua, key=lambda t: t[2])
rbf = GridSearchCV(SVC(kernel='rbf'),
                   {'C': [0.1, 1, 10, 100], 'gamma': [0.01, 0.1, 1, 10]}, cv=5)
rbf.fit(X_tr, y_tr)
print(f'Poly tot nhat theo CV: bac {best_poly[0]}, CV={best_poly[2]:.4f}')
print(f'RBF muc 5: CV={rbf.best_score_:.4f}, Test={rbf.score(X_te, y_te):.4f}')
print('Poly vuot RBF theo CV:', best_poly[2] > rbf.best_score_)


Poly bac 2: C=1, CV=0.5422, Test=0.5067
Poly bac 3: C=1, CV=0.8178, Test=0.8267


Poly bac 4: C=100, CV=0.5156, Test=0.4800


Poly tot nhat theo CV: bac 3, CV=0.8178
RBF muc 5: CV=0.8711, Test=0.9200
Poly vuot RBF theo CV: False


**Nhận xét**

- Kernel đa thức bậc 3 tốt nhất trong ba bậc thử, CV đạt 81,78%. Kernel RBF vẫn cao hơn với CV 87,11% và Accuracy test 92%.

### Bài tập 4 - Dò lưới mịn C và Gamma

In [9]:
import pandas as pd
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.svm import SVC

df = pd.read_csv('data/hai_vang_trang.csv')
X, y = df[['x1', 'x2']], df['nhan']
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

# Lưới gốc đúng như mục 5
cu = GridSearchCV(SVC(kernel='rbf'),
                  {'C': [0.1, 1, 10, 100], 'gamma': [0.01, 0.1, 1, 10]}, cv=5)
cu.fit(X_tr, y_tr)

# Lưới mới tập trung gần vùng tham số tốt
moi = GridSearchCV(SVC(kernel='rbf'),
                   {'C': [3, 10, 30], 'gamma': [0.3, 1, 3]}, cv=5)
moi.fit(X_tr, y_tr)
print(f'Luoi goc: {cu.best_params_}, CV={cu.best_score_:.4f}, Test={cu.score(X_te, y_te):.4f}')
print(f'Luoi min: {moi.best_params_}, CV={moi.best_score_:.4f}, Test={moi.score(X_te, y_te):.4f}')
print('Diem CV vuot 0.8711:', moi.best_score_ > 0.8711)
print('Accuracy test co doi:', abs(moi.score(X_te, y_te) - cu.score(X_te, y_te)) > 1e-12)


Luoi goc: {'C': 10, 'gamma': 1}, CV=0.8711, Test=0.9200
Luoi min: {'C': 30, 'gamma': 1}, CV=0.8756, Test=0.9200
Diem CV vuot 0.8711: True
Accuracy test co doi: False


**Nhận xét**

- Lưới mịn đạt CV 87,56%, cao hơn mức cũ 87,11%, nhưng Accuracy trên test vẫn là 92%. Tăng số lần thử không chắc cải thiện được kết quả thực tế.

### Bài tập 5 - Ma trận nhầm lẫn và ảnh chữ số dự đoán sai

In [10]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

X, y = load_digits(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)
svm = SVC(kernel='rbf').fit(X_tr, y_tr)
y_pred = svm.predict(X_te)
cm = confusion_matrix(y_te, y_pred, labels=np.arange(10))
print('Accuracy:', round(accuracy_score(y_te, y_pred), 4))
print('Ma tran nham lan (hang that, cot du doan):')
print(cm)

# Liệt kê các cặp chữ số bị nhầm
for a in range(10):
    for b in range(10):
        if a != b and cm[a, b] > 0:
            print(f'  Chu so {a} bi nham thanh {b}: {cm[a, b]} lan')

# Lưu từng ảnh bị đoán sai để dễ kiểm tra
thu_muc = Path('figures') / 'anh_chu_so_sai'
thu_muc.mkdir(parents=True, exist_ok=True)
chi_so_sai = np.where(y_pred != y_te)[0]
for i in chi_so_sai:
    plt.figure(figsize=(2.4, 2.4))
    plt.imshow(X_te[i].reshape(8, 8), cmap='gray_r')
    plt.title(f'That: {y_te[i]} | Doan: {y_pred[i]}')
    plt.axis('off')
    plt.tight_layout()
    plt.savefig(thu_muc / f'anh_{i:03d}_that_{y_te[i]}_doan_{y_pred[i]}.png', dpi=130)
    plt.close()
print('So anh du doan sai:', len(chi_so_sai))
print('Da luu anh trong:', thu_muc)


Accuracy: 0.9911
Ma tran nham lan (hang that, cot du doan):
[[45  0  0  0  0  0  0  0  0  0]
 [ 0 46  0  0  0  0  0  0  0  0]
 [ 0  0 44  0  0  0  0  0  0  0]
 [ 0  0  0 46  0  0  0  0  0  0]
 [ 0  0  0  0 44  0  0  0  1  0]
 [ 0  0  0  0  0 46  0  0  0  0]
 [ 0  0  0  0  0  0 45  0  0  0]
 [ 0  0  0  0  0  0  0 44  0  1]
 [ 0  1  0  0  0  0  0  0 42  0]
 [ 0  0  0  1  0  0  0  0  0 44]]
  Chu so 4 bi nham thanh 8: 1 lan
  Chu so 7 bi nham thanh 9: 1 lan
  Chu so 8 bi nham thanh 1: 1 lan
  Chu so 9 bi nham thanh 3: 1 lan
So anh du doan sai: 4
Da luu anh trong: figures/anh_chu_so_sai


**Nhận xét**

- SVM đạt Accuracy 99,11% trên chữ số viết tay và nhầm 4 ảnh: 4 thành 8, 7 thành 9, 8 thành 1, 9 thành 3. Nét viết tương tự nhau và ảnh 8×8 ít chi tiết có thể khiến mô hình nhầm.

### Bài tập 6 - So sánh tốc độ SVC và LinearSVC

In [11]:
import time
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.svm import SVC, LinearSVC

hang = []
for n in [2000, 5000, 10000, 20000]:
    X, y = make_classification(n_samples=n, n_features=20, random_state=0)
    for ten, mh in [('SVC', SVC()), ('LinearSVC', LinearSVC(random_state=0, max_iter=5000))]:
        t0 = time.perf_counter()
        mh.fit(X, y)
        so_giay = time.perf_counter() - t0
        hang.append({'So mau': n, 'Mo hinh': ten, 'Thoi gian (giay)': so_giay})
        print(f'{n:6d} | {ten:9s} | {so_giay:8.4f} giay')

bang = pd.DataFrame(hang).pivot(index='So mau', columns='Mo hinh', values='Thoi gian (giay)')
print('\nBang so sanh thoi gian:')
print(bang.round(4).to_string())
print('\nMuc tang so voi lan truoc:')
print((bang / bang.shift(1)).round(2).to_string())


  2000 | SVC       |   0.0229 giay
  2000 | LinearSVC |   0.0021 giay


  5000 | SVC       |   0.1210 giay
  5000 | LinearSVC |   0.0048 giay


 10000 | SVC       |   1.5407 giay
 10000 | LinearSVC |   0.0079 giay


 20000 | SVC       |   4.0917 giay
 20000 | LinearSVC |   0.0159 giay

Bang so sanh thoi gian:
Mo hinh  LinearSVC     SVC
So mau                    
2000        0.0021  0.0229
5000        0.0048  0.1210
10000       0.0079  1.5407
20000       0.0159  4.0917

Muc tang so voi lan truoc:
Mo hinh  LinearSVC    SVC
So mau                   
2000           NaN    NaN
5000          2.22   5.29
10000         1.65  12.73
20000         2.02   2.66


**Nhận xét**

- SVC thường tốn nhiều thời gian hơn LinearSVC khi tăng số mẫu. Có thể xem tỷ số giữa các lần chạy trong bảng; thời gian thực tế thay đổi theo cấu hình máy.